# **Tech Challenge Fase 2 - Indicador Criança Alfabetizada**
## 🥉 Ingestão Bronze · Download dos dados brutos

**Objetivo:** baixar as tabelas oficiais da Base dos Dados **exatamente como vêm** (sem filtro e sem transformação) e **acrescentá-las** nas tabelas da camada Bronze, no dataset `bronze_tc` do BigQuery.

| Etapa | Descrição |
|---|---|
| 1. Setup | Configurações |
| 2. Conexão | Login e conferência do dataset Bronze |
| 3. Função de ingestão | Baixa, confere e carrega no BigQuery |
| 4. Execução | Roda para todas as tabelas |
| 5. Conferência | Compara as linhas da origem com as carregadas |

---
## Etapa 1 — Setup

In [ ]:

# ============================================================
# IMPORTS E CONFIGURAÇÕES GLOBAIS
# ============================================================

# ─── Imports ───
import os
import gc
from datetime import datetime, timezone

import pandas as pd
from google.colab import auth
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

# ─── Configurações (lidas do .env, se existir; senão valem os padrões abaixo) ───
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

PROJECT_ID = os.getenv("GCP_PROJECT_ID", "project-ce782257-1c9c-4b6f-87a")
BRONZE_DATASET = os.getenv("BQ_BRONZE_DATASET", "bronze_tc")

DATASET_ORIGEM = os.getenv("SOURCE_DATASET", "basedosdados.br_inep_avaliacao_alfabetizacao")
TABELAS = ["uf", "municipio", "meta_alfabetizacao_brasil", "meta_alfabetizacao_uf", "meta_alfabetizacao_municipio", "dicionario", "alunos"]
LIMITE_GB = float(os.getenv("LIMITE_GB", "2"))  # Trava de segurança por tabela

# ─── Metadados de ingestão ───
INGESTION_TS = datetime.now(timezone.utc)
print("Timestamp de ingestão:", INGESTION_TS.isoformat())

---
## Etapa 2 — Conexão com o BigQuery

In [ ]:
auth.authenticate_user()

client = bigquery.Client(project=PROJECT_ID)
bronze = client.get_dataset(f"{PROJECT_ID}.{BRONZE_DATASET}")
print("Conectado ao projeto:", client.project)
print("Dataset Bronze:", bronze.dataset_id, "| região:", bronze.location)

---
## Etapa 3 — Função de ingestão

Para cada tabela: baixa, confere `shape` e `info`, acrescenta duas colunas de controle e **acrescenta** as linhas na tabela de mesmo nome em `bronze_tc`.

In [ ]:
def ingerir_bronze(tabela: str) -> dict:
    """
    Baixa uma tabela da origem sem filtro e sem transformação e acrescenta na Bronze do BigQuery.

    Args:
        tabela: Nome da tabela de origem.

    Returns:
        dict: Status da ingestão, contagem de linhas, checagem de contagem e
        ID do job de carga no BigQuery.
    """

    ref = f"{DATASET_ORIGEM}.{tabela}"
    destino = f"{PROJECT_ID}.{BRONZE_DATASET}.{tabela}"
    info_origem = client.get_table(ref)
    gb = info_origem.num_bytes / 1e9
    print(f"== {tabela} ==")
    print(f"   origem: {info_origem.num_rows} linhas, {gb:.3f} GB")

    # Trava de segurança: tabela grande demais pode derrubar a memória do Colab
    if gb > LIMITE_GB:
        print(f"   PULADA: acima do limite de {LIMITE_GB} GB")
        return {"tabela": tabela, "status": "PULADA", "linhas_origem": info_origem.num_rows}

    df = client.list_rows(ref).to_dataframe()

    # ── Checagens ──
    print(f"   shape: {df.shape}")
    df.info()

    # ── Metadados de ingestão (colunas novas; os dados originais não mudam) ──
    df["_ingestion_timestamp"] = pd.Timestamp(INGESTION_TS)
    df["_source"] = ref

    # ── Schema: mesmos nomes e tipos da origem + as 2 colunas de controle ──
    schema = [bigquery.SchemaField(c.name, c.field_type) for c in info_origem.schema]
    schema += [
        bigquery.SchemaField("_ingestion_timestamp", "TIMESTAMP"),
        bigquery.SchemaField("_source", "STRING"),
    ]

    # ── Contagem antes da carga ──
    try:
        antes = client.get_table(destino).num_rows
    except NotFound:
        antes = 0
    
    # ── Carga na Bronze (acrescenta; cria a tabela se ainda não existir) ──
    job_config = bigquery.LoadJobConfig(schema=schema, write_disposition="WRITE_APPEND")
    job = client.load_table_from_dataframe(df, destino, job_config=job_config)
    job.result()

    depois = client.get_table(destino).num_rows
    linhas_baixadas = len(df)
    del df
    gc.collect()  # libera memória antes da próxima tabela

    return {
        "tabela": tabela,
        "status": "OK",
        "linhas_origem": info_origem.num_rows,
        "linhas_baixadas": linhas_baixadas,
        "linhas_antes": antes,
        "linhas_depois": depois,
        "linhas_acrescentadas": depois - antes,
        "checagem": info_origem.num_rows == linhas_baixadas == depois - antes,
        "job_id": job.job_id,
        "duracao_s": round((job.ended - job.started).total_seconds(), 1),
    }

---
## Etapa 4 — Execução

In [ ]:
resultados = []
for tabela in TABELAS:
    resultados.append(ingerir_bronze(tabela))
    print()

---
## Etapa 5 — Conferência

A coluna `checagem` precisa ser `True` em todas as tabelas: linhas na origem = linhas baixadas = linhas acrescentadas na Bronze.

In [ ]:
df_resultados = pd.DataFrame(resultados)
df_resultados.to_csv("evidencias_bronze.csv", index=False)
display(df_resultados)

divergentes = df_resultados[df_resultados["checagem"] == False]
if len(divergentes) > 0:
    raise ValueError(f"Contagem de linhas diverge em: {list(divergentes['tabela'])}")

pulados = df_resultados[df_resultados["status"] == "PULADA"]
if len(pulados) > 0:
    print("Atenção, tabelas puladas:", list(pulados["tabela"]))

print("\nTabelas hoje na Bronze:")
for t in client.list_tables(f"{PROJECT_ID}.{BRONZE_DATASET}"):
    print(f"  {t.table_id}: {client.get_table(t).num_rows} linhas")